# AdS4 Dual-Solution Probe Fit Test (v16)

Goal: test whether two different reconstructed geometries can fit the same probe stack equally well.

This notebook extends v15 by:
- training two solutions under different GEO constraints (A and B)
- evaluating both solutions against the same observable set
- comparing per-probe residuals (EE, WL, GEO_A, GEO_B)
- visualizing whether the solutions are distinguishable by the current probes

Interpretation:
- if both solutions fit the shared probes similarly, the inverse problem is non-unique under that probe set
- if one solution clearly fails some probes, the constraint basis is selecting between distinguishable geometries


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee(f, g):
    return torch.log(1 + 0.6 * f)

def wl(f, g):
    return torch.sqrt(f + 1.8 * g + 1e-6)

def geo(f, g):
    return torch.sqrt(f + 0.7 * g + 1e-6)

ee_obs = ee(f_true(r), g_true(r)).detach()
wl_obs = wl(f_true(r), g_true(r)).detach()
geo_base = geo(f_true(r), g_true(r)).detach()


In [ ]:
def geo_A(r):
    return geo_base

def geo_B(r):
    return geo_base * (1 + 0.25 * (r - r.min()) / (r.max() - r.min())) * (1 + 0.1 * torch.sin(2 * r))


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()
    def forward(self, r):
        return F.softplus(self.f(r)), F.softplus(self.g(r))


In [ ]:
def train(geo_target_fn, seed=0, epochs=1200, consistency_weight=0.15):
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    target = geo_target_fn(r)
    loss_hist = []
    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)
        loss = ((ee(f,g)-ee_obs)**2).mean() + ((wl(f,g)-wl_obs)**2).mean() + ((geo(f,g)-target)**2).mean() + 0.15*((f*g-1)**2).mean()
        loss.backward()
        opt.step()
        loss_hist.append(loss.item())
    with torch.no_grad():
        f_pred, g_pred = m(r)
    return {'f': f_pred.detach().cpu(), 'g': g_pred.detach().cpu(), 'loss_hist': loss_hist}

run_A = train(geo_A, seed=0)
run_B = train(geo_B, seed=0)

In [ ]:
# Evaluate both solutions against both probe sets
with torch.no_grad():
    fA, gA = run_A['f'].to(device), run_A['g'].to(device)
    fB, gB = run_B['f'].to(device), run_B['g'].to(device)

    probes_A = {
        'EE': ee(fA, gA),
        'WL': wl(fA, gA),
        'GEO_A': geo(fA, gA),
        'GEO_B': geo(fA, gA),
    }
    probes_B = {
        'EE': ee(fB, gB),
        'WL': wl(fB, gB),
        'GEO_A': geo(fB, gB),
        'GEO_B': geo(fB, gB),
    }

    targets = {
        'EE': ee_obs,
        'WL': wl_obs,
        'GEO_A': geo_A(r),
        'GEO_B': geo_B(r),
    }

def mse(pred, target):
    return float(((pred - target)**2).mean().item())

rows = []
for sol_name, probes in [('A', probes_A), ('B', probes_B)]:
    for probe_name, pred in probes.items():
        rows.append((sol_name, probe_name, mse(pred, targets[probe_name])))

rows

In [ ]:
# Residual matrix for plotting
probe_names = ['EE', 'WL', 'GEO_A', 'GEO_B']
sol_names = ['A', 'B']
resid = np.zeros((len(sol_names), len(probe_names)))
for i, s in enumerate(sol_names):
    for j, p in enumerate(probe_names):
        resid[i, j] = [x[2] for x in rows if x[0] == s and x[1] == p][0]
resid

In [ ]:
# Main figure: geometry + residual comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(r.cpu(), f_true(r).cpu(), 'k', label='true')
axes[0,0].plot(r.cpu(), run_A['f'], label='A')
axes[0,0].plot(r.cpu(), run_B['f'], label='B')
axes[0,0].set_title('f(r): dual solutions')
axes[0,0].legend()

axes[0,1].plot(r.cpu(), g_true(r).cpu(), 'k', label='true')
axes[0,1].plot(r.cpu(), run_A['g'], label='A')
axes[0,1].plot(r.cpu(), run_B['g'], label='B')
axes[0,1].set_title('g(r): dual solutions')
axes[0,1].legend()

im = axes[1,0].imshow(resid, aspect='auto')
axes[1,0].set_xticks(range(len(probe_names)))
axes[1,0].set_xticklabels(probe_names)
axes[1,0].set_yticks(range(len(sol_names)))
axes[1,0].set_yticklabels(sol_names)
axes[1,0].set_title('per-probe MSE')
fig.colorbar(im, ax=axes[1,0], fraction=0.046, pad=0.04)

x = np.arange(len(probe_names))
w = 0.35
axes[1,1].bar(x - w/2, resid[0], width=w, label='solution A')
axes[1,1].bar(x + w/2, resid[1], width=w, label='solution B')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(probe_names)
axes[1,1].set_title('probe-fit comparison')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('ads4_phase_lock_v16_probe_fit.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v16_probe_fit.png')

In [ ]:
# Training trajectories
plt.figure(figsize=(8,4))
plt.plot(run_A['loss_hist'], label='solution A training')
plt.plot(run_B['loss_hist'], label='solution B training')
plt.title('training trajectories')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.tight_layout()
plt.savefig('ads4_phase_lock_v16_losses.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v16_losses.png')